# Import dependencies

In [1]:
import numpy as np
import pandas as pd

print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")

NumPy: 2.5.4
pandas: 3.1.1


# Load dataset

In [2]:
# Since the dataset file is saved in the Strict Open XML Spreadsheet format, we
# have to use `calamine` to read it
df = pd.read_excel("dataset/dataset-uci.xlsx", engine="calamine")

# Shape of dataset; data types; feature overview; missing values; duplicates; etc.
rows = len(df)
print(f"Number of rows: {rows}")
#how many columns we have of data
columns = len(df.columns)
print(f"Number of columns: {columns}")
##Now the actual shape of the dataset + types
print(f"Dataset shape: {df.shape}")
print(f"Dataset datatypes:\n{df.dtypes}")

print(f"# null values for each feature:\n{df.isnull().sum()}")
print(f"# null values: {df.isnull().sum().sum()}")

print(f"# duplicated values: {df.duplicated().sum()}")

y = df["Gallstone Status"]
X = df.drop(columns="Gallstone Status")

Number of rows: 319
Number of columns: 39
Dataset shape: (319, 39)
Dataset datatypes:
Gallstone Status                                    int64
Age                                                 int64
Gender                                              int64
Comorbidity                                         int64
Coronary Artery Disease (CAD)                       int64
Hypothyroidism                                      int64
Hyperlipidemia                                      int64
Diabetes Mellitus (DM)                              int64
Height                                              int64
Weight                                            float64
Body Mass Index (BMI)                             float64
Total Body Water (TBW)                            float64
Extracellular Water (ECW)                         float64
Intracellular Water (ICW)                         float64
Extracellular Fluid/Total Body Water (ECF/TBW)    float64
Total Body Fat Ratio (TBFR) (%)             

We have data of around 319 ppl.

In [3]:
#How many ppl have gallstones and how many do not
gallstone_counts = y.value_counts()[1]
print(f"Number of people with gallstones: {gallstone_counts}")
print(f"Number of people without gallstones: {len(X) - gallstone_counts}")


Number of people with gallstones: 158
Number of people without gallstones: 161


So we have half half ppl with and without gallstones

In [4]:
# Outlier
v_max =  X.max()
v_mean = X.mean()
v_min =  X.min()
v_std =  X.std()

z = (X - v_mean) / v_std
outliers = z.abs() > 10

print(outliers.sum())               # number of outliers per column
result = pd.DataFrame({
    "value": X[outliers].stack().dropna(),
    "z": z[outliers].stack().dropna(),
})

feature = result.index.get_level_values(1)
result["mean"] = v_mean[feature].values
result["std"] = v_std[feature].values

result = result[["value", "mean", "std", "z"]].rename_axis(["Record", "Feature"])
print(result.round(2))

Age                                               0
Gender                                            0
Comorbidity                                       0
Coronary Artery Disease (CAD)                     0
Hypothyroidism                                    0
Hyperlipidemia                                    0
Diabetes Mellitus (DM)                            0
Height                                            0
Weight                                            0
Body Mass Index (BMI)                             0
Total Body Water (TBW)                            0
Extracellular Water (ECW)                         0
Intracellular Water (ICW)                         0
Extracellular Fluid/Total Body Water (ECF/TBW)    0
Total Body Fat Ratio (TBFR) (%)                   0
Lean Mass (LM) (%)                                0
Body Protein Content (Protein) (%)                0
Visceral Fat Rating (VFR)                         0
Bone Mass (BM)                                    0
Muscle Mass 

In [ ]:
# This is very high but clinically possible in poorly controlled diabetes
X.loc[40, "Diabetes Mellitus (DM)"]

np.int64(0)

In [ ]:
# This isn’t physiologically possible. It may be a misplaced decimal, such as 19.54 or 195.4. Obesity % is usually derived from BMI
X.loc[239, ["Height", "Weight", "Body Mass Index (BMI)", "Obesity (%)"]]
X.loc[239, "Obesity (%)"] = 19.54

In [ ]:
X.loc[318, ["Total Cholesterol (TC)", "Low Density Lipoprotein (LDL)", "High Density Lipoprotein (HDL)"]]
X = X.drop(318)
y = y.drop(318)

In [ ]:
# Highly correlated features
corr = X.corr().abs()
upper = corr.where(np.triu(
    np.ones(corr.shape, dtype=bool),
    k=1
))  # upper triangle only

pairs = upper.stack()
high = pairs[pairs > 0.9]
print(high.sort_values(key=np.abs, ascending=False))


Total Body Fat Ratio (TBFR) (%)  Lean Mass (LM) (%)           0.994052
Total Body Water (TBW)           Muscle Mass (MM)             0.928289
Body Mass Index (BMI)            Obesity (%)                  0.917637
Total Body Water (TBW)           Extracellular Water (ECW)    0.904057
Intracellular Water (ICW)        Muscle Mass (MM)             0.903617
Body Mass Index (BMI)            Total Fat Content (TFC)      0.902664
dtype: float64


In [ ]:
keep = []
for c in X.columns:
    if not (corr.loc[c, keep] > 0.9).any():
        keep.append(c)

dropped = [c for c in X.columns if c not in keep]
print(f"Dropped {len(dropped)}:", dropped)

X = X[keep]

Dropped 5: ['Extracellular Water (ECW)', 'Lean Mass (LM) (%)', 'Muscle Mass (MM)', 'Obesity (%)', 'Total Fat Content (TFC)']


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# 1. Hold out a test set — not touched until the very end
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

def make_model(C):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=C, max_iter=5000),
    )

# 2. Choose C with cross-validation on the training set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    scores = cross_validate(make_model(C), X_train, y_train, cv=cv,
                            scoring=["accuracy", "roc_auc", "f1"])
    results[C] = scores["test_roc_auc"].mean()

    print(f"C={C}:")
    for metric in ["accuracy", "roc_auc", "f1"]:
        s = scores[f"test_{metric}"]
        print(f"{metric:>8}: {s.mean():.3f} ± {s.std():.3f}")

best_C = max(results, key=results.get)
print(f"\nBest C (by CV ROC AUC): {best_C}")

# 3. Refit on the full training set and evaluate once on the test set
final_model = make_model(best_C)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print("\nTest set performance:")
print(f"accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f" roc_auc: {roc_auc_score(y_test, y_prob):.3f}")
print(f"      f1: {f1_score(y_test, y_pred):.3f}")

C=0.001:
accuracy: 0.638 ± 0.051
 roc_auc: 0.732 ± 0.051
      f1: 0.599 ± 0.058
C=0.01:
accuracy: 0.724 ± 0.035
 roc_auc: 0.808 ± 0.046
      f1: 0.699 ± 0.028
C=0.1:
accuracy: 0.764 ± 0.022
 roc_auc: 0.852 ± 0.036
      f1: 0.746 ± 0.015
C=1:
accuracy: 0.776 ± 0.044
 roc_auc: 0.871 ± 0.030
      f1: 0.763 ± 0.036
C=10:
accuracy: 0.787 ± 0.040
 roc_auc: 0.874 ± 0.027
      f1: 0.775 ± 0.039
C=100:
accuracy: 0.772 ± 0.054
 roc_auc: 0.855 ± 0.036
      f1: 0.759 ± 0.055
C=1000:
accuracy: 0.768 ± 0.053
 roc_auc: 0.842 ± 0.042
      f1: 0.758 ± 0.050

Best C (by CV ROC AUC): 10

Test set performance:
accuracy: 0.781
 roc_auc: 0.854
      f1: 0.781


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

svm = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),          # essential for SVMs
    SVC(),
)

param_grid = [
    {"svc__kernel": ["linear"],
     "svc__C": [0.001, 0.01, 0.1, 1, 10]},
    {"svc__kernel": ["rbf"],
     "svc__C": [0.1, 1, 10, 100],
     "svc__gamma": ["scale", 0.001, 0.01, 0.1]},
]

grid = GridSearchCV(svm, param_grid, cv=cv, scoring="roc_auc")
grid.fit(X_train, y_train)

i = grid.best_index_
print("Best params:", grid.best_params_)
print(f"CV ROC AUC: {grid.cv_results_['mean_test_score'][i]:.3f} "
      f"± {grid.cv_results_['std_test_score'][i]:.3f}")

# Final evaluation on the test set
best = grid.best_estimator_
y_pred = best.predict(X_test)
y_score = best.decision_function(X_test)

print("\nTest set performance:")
print(f"accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f" roc_auc: {roc_auc_score(y_test, y_score):.3f}")
print(f"      f1: {f1_score(y_test, y_pred):.3f}")

Best params: {'svc__C': 1, 'svc__kernel': 'linear'}
CV ROC AUC: 0.887 ± 0.030

Test set performance:
accuracy: 0.781
 roc_auc: 0.857
      f1: 0.767


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

skewed = [c for c in ["C-Reactive Protein (CRP)", "Aspartat Aminotransferaz (AST)",
                      "Alanin Aminotransferaz (ALT)", "Triglyceride"] if c in X.columns]

models = {
    "logreg": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=10, max_iter=5000),
    ),
    "logreg + log": make_pipeline(
        ColumnTransformer([("log", FunctionTransformer(np.log1p), skewed)],
                          remainder="passthrough"),
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=1, max_iter=5000),
    ),
    "random forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=500, random_state=42),
    ),
    "gradient boosting": HistGradientBoostingClassifier(   # handles NaN natively
        max_depth=3, learning_rate=0.05, max_iter=200, random_state=42),
}

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for name, model in models.items():
    s = cross_val_score(model, X_train, y_train, cv=rcv, scoring="roc_auc")
    print(f"{name:>18}: {s.mean():.3f} ± {s.std():.3f}")

            logreg: 0.856 ± 0.041
      logreg + log: 0.888 ± 0.038
     random forest: 0.865 ± 0.045
 gradient boosting: 0.850 ± 0.047


In [13]:
from sklearn.inspection import permutation_importance

model = models["random forest"]
model.fit(X_train, y_train)

r = permutation_importance(model, X_test, y_test,
                           scoring="roc_auc", n_repeats=30, random_state=42)

imp = pd.DataFrame({"mean": r.importances_mean, "std": r.importances_std},
                   index=X_test.columns).sort_values("mean", ascending=False)
print(imp.head(15))

                                                    mean       std
C-Reactive Protein (CRP)                        0.098079  0.032977
Vitamin D                                       0.045687  0.023462
Extracellular Fluid/Total Body Water (ECF/TBW)  0.012109  0.011623
Body Protein Content (Protein) (%)              0.006559  0.006586
Bone Mass (BM)                                  0.004541  0.015820
Alkaline Phosphatase (ALP)                      0.003564  0.003669
Intracellular Water (ICW)                       0.002148  0.003743
Triglyceride                                    0.002018  0.002450
Visceral Fat Area (VFA)                         0.001872  0.003892
Gender                                          0.000830  0.001359
Total Body Water (TBW)                          0.000716  0.003640
Hyperlipidemia                                  0.000358  0.001696
Coronary Artery Disease (CAD)                   0.000114  0.000483
Hemoglobin (HGB)                                0.000000  0.00